# 03 — Match parents to children in SQL and estimate the IGE

**Reads:** `data/processed/psid.sqlite`. **Writes:** `data/processed/matched.csv` (one row per child with parent
and child income measures), `results/tables/ige_estimates.csv`, `results/tables/ige_by_child_age.csv`.

All matching and aggregation happens in SQL run through `sqlite3`; pandas only receives results.
Every query in the post is copied from here, so write them as named strings (or `.sql` files in `code/sql/`).

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
PROC, TAB = ROOT / "data" / "processed", ROOT / "results" / "tables"
conn = sqlite3.connect(PROC / "psid.sqlite")

def q(sql, **params):
    return pd.read_sql_query(sql, conn, params=params)

## 1. SELECT / WHERE — the child sample
Define who counts as a "child" observation: a person-year in which the person is head or spouse of their own
family unit (`rel_head`), within the age window you chose. Count them.

## 2. JOIN — person-years to family income
`persons ⋈ families` on (`year`, `interview` = `fam_interview`). Inner join, with the row count before and after.

## 3. JOIN through identifiers — child to father (and/or mother)
`individuals` gives each child the father's (`id_1968`, `person_num`). Join that pair back to the father's own
person-years. Use a LEFT JOIN first and count the NULLs: that is the number of children whose father is not in the
study, and it goes in the post's caveats.

## 4. GROUP BY — parent income, single-year and multi-year
Parent income = father's real family income in the years you chose (the professor's suggestion: 2–3 years is enough).
Produce both: one year, and the average over the years. `GROUP BY` the child; `AVG()` the income; `COUNT()` the years actually observed.

## 5. The IGE
$$\ln y_{\text{child}} = \alpha + \beta \ln y_{\text{parent}} + \varepsilon$$
$\beta$ is the intergenerational elasticity. Estimate it (a) with single-year parent income, (b) with the multi-year average,
and (c) separately by the child's age at measurement (e.g. 25, 30, 35, 40, 45) — that last one is the life-cycle bias figure.
Save every estimate with its standard error and N to `results/tables/`.